# 泛型与类型参数

学习目标：能用泛型保留输入输出关系，并正确标注集合、异步结果和迭代协议。

前置知识：TypeScript 函数、联合、结构兼容和元组；JavaScript Promise、Map、Set、生成器与异步迭代。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict。本章附加选项：noUncheckedIndexedAccess=true，含义见对应知识点。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/09-generics/。

1. [main.ts](scripts/09-generics/main.ts)：按正文顺序组织的正常示例，片段依赖同文件前文定义。
2. [type-errors.ts](scripts/09-generics/type-errors.ts)：与正常示例隔离的类型反例，不生成或执行 JavaScript。
3. [tsconfig.json](scripts/09-generics/tsconfig.json)、[tsconfig.errors.json](scripts/09-generics/tsconfig.errors.json)：分别明确正常与反例文件范围。



Step 1：检查正常项目的类型。

```bash
npm run check:09
```

Step 2：生成正常项目的 JavaScript。

```bash
npm run build:09
```

Step 3：运行正常示例。

```bash
npm run run:09
```

Step 4：检查下文独立列出的类型反例。

```bash
npm run errors:09
# 预期非零退出；按反例注释逐行核对具体错误，不运行 type-errors.ts。
```

正常配置只包含上面列出的正常与独立运行示例，生成文件位于 .build/09-generics/。错误配置继承正常选项，改用 type-errors.ts 并开启 noEmit。

## 1 类型参数保留输入与输出关系

泛型的作用是保留关系：一次调用选定的类型，必须贯穿参数与返回值。

泛型（generics）让一个实现适用于多种类型，同时保留它们之间的关系。T 是类型参数，代表本次调用选定的类型；调用 identity&lt;string&gt; 中的 string 是类型实参。通常可以从实际参数推断，只有意图不清或需主动指定范围时再显式传类型。

unknown 可接收所有值，但返回 unknown 会丢失调用者已经知道的类型；any 则放宽检查。identity 的返回 T 表明结果仍具有输入类型。泛型参数不是运行时构造函数，不能在函数体内当作值来 new。

![泛型把输入与输出的类型连接起来。T 是本次调用的类型参数，不是运行时的变量或构造函数。](image/illustration/09-01-generic-input-output.svg)

图示说明：依据泛型输入输出关系自绘，图使用 identity 的一个类型参数；pair 则让同一 T 同时约束两个输入。

下面先观察显式 number 调用，再检查返回结果可以使用哪些方法；最后判断 same&lt;number&gt;("3") 在关系的哪一端冲突。

```typescript
export {};
function identity<T>(value: T): T { return value; }
function pair<T>(left: T, right: T): [T, T] { return [left, right]; }
const text = identity("泛型");
const count = identity<number>(3);
const mixed = pair<string | number>("一", 1);
console.log(text.toUpperCase(), count.toFixed(0), mixed.join(",")); // 泛型 3 一,1
```

以下片段来自独立的 type-errors.ts：

```typescript
function same<T>(value: T): T { return value; }
same<number>("3"); // 显式类型实参要求 number。
function createFromType<T>(): T { return new T(); } // T 仅存在于类型位置，不是构造函数值。
```

## 2 泛型接口、别名与默认值

接口和类型别名也可以接收类型参数。Box&lt;T&gt; 的一次实例化决定其整个结构中 T 的含义；相反，接口里的泛型调用签名 &lt;T&gt;(value: T): T 允许每次调用分别选择 T。

默认类型实参让调用者省略对应类型参数。默认值必须满足约束，有默认值的参数后不能再出现必需类型参数；有推断候选时先按推断决定，而不是一律使用默认值。默认类型不会创建运行时默认对象。

```typescript
interface Box<T = string> { value: T; }
type Maybe<T> = T | undefined;
interface Identity { <T>(value: T): T; }
const genericIdentity: Identity = identity;
const titleBox: Box = { value: "类型" };
const numberBox: Box<number> = { value: 2 };
const absent: Maybe<number> = undefined;
console.log(titleBox.value, numberBox.value, genericIdentity(true), absent); // 类型 2 true undefined
```

以下片段来自独立的 type-errors.ts：

```typescript
interface DefaultBox<T = string> { value: T; }
const wrongDefault: DefaultBox = { value: 3 }; // 省略实参时默认 T 为 string。
```

## 3 约束提供能力，不替代具体类型

T extends { length: number } 约束可接受的类型至少有数值 length。函数体可以访问这项能力，并仍返回完整 T；字符串、数组和有额外字段的对象都可以符合这个结构。

约束不是把 T 固定成约束对象。调用者可能传入还有其他必需成员的类型，因此实现不能随意构造一个只有 length 的对象并声称它就是 T。下面 first 返回 T | undefined，明确空数组可能没有首项；本章也开启 noUncheckedIndexedAccess。

```typescript
function keepLength<T extends { length: number }>(value: T): T { return value; }
function first<T>(values: readonly T[]): T | undefined { return values[0]; }
const detailed = keepLength({ length: 2, label: "课" });
const firstText = first(["A", "B"]);
console.log(detailed.label, firstText?.toLowerCase(), first<number>([])); // 课 a undefined
```

以下片段来自独立的 type-errors.ts：

```typescript
function constrained<T extends { length: number }>(value: T): T { return value; }
constrained(3); // number 没有 length。
function fabricate<T extends { length: number }>(): T {
  return { length: 0 }; // T 可能还要求其他成员，满足约束不等于满足任意 T。
}
```

## 4 const 类型参数与可变元组展开

const 类型参数自 TypeScript 5.0 起让调用位置的对象、数组和原始字面量采用更精确的 const 式推断。为了接收推断出的只读元组，约束也要允许只读数组；可写数组约束可能使推断退回较宽类型。

它不会把先前已拓宽的变量重新还原为字面量，也不会冻结运行时数据。可变元组类型（variadic tuple types）中的 ...A 与 ...B 则把两组位置类型展开到新元组；A、B 分别代表输入的数组或元组类型。它保留顺序与已知长度，而不只是联合所有元素类型。

```typescript
function keep<const T extends readonly string[]>(values: T): T { return values; }
const exact = keep(["读", "写"]); // readonly ["读", "写"]。
const existing = ["读", "写"];
const broad = keep(existing); // string[]：已有变量的类型不会倒推恢复。
function join<A extends readonly unknown[], B extends readonly unknown[]>(left: A, right: B): [...A, ...B] {
  return [...left, ...right];
}
const joined = join(["TS", 1] as const, [true] as const);
const fixed: ["TS", 1, true] = joined;
console.log(exact[0], broad.length, fixed.join(",")); // 读 2 TS,1,true
```

以下片段来自独立的 type-errors.ts：

```typescript
function keepLiteral<const T extends readonly string[]>(values: T): T { return values; }
const exactLiteral = keepLiteral(["读", "写"]);
exactLiteral[0] = "读"; // 推断的元组位置为 readonly。
```

## 5 Promise、Map 与 Set 的类型关系

标准库把许多容器描述为泛型：Promise&lt;T&gt; 的 T 是兑现后的值类型，不是拒绝原因类型；Map&lt;K, V&gt; 的 K 和 V 分别是键和值类型；Set&lt;T&gt; 的 T 是元素类型。这些声明不会为宿主安装实现，当前 Node 环境提供相应 JavaScript 对象。

async 函数应返回 Promise 包装的类型，await 取得兑现值。Map.get 即使键类型正确也可能找不到键，结果包含 undefined；Set 的类型参数不改变运行时的去重规则。这里仅使用本地已知数据，不请求网络。

```typescript
async function ready<T>(value: T): Promise<T> { return value; }
const promised: Promise<number> = ready(7);
const prices = new Map<string, number>([["笔记", 3]]);
const tags = new Set<string>(["类型", "类型", "函数"]);
const resolved = await promised;
console.log(resolved.toFixed(0), prices.get("缺失") ?? 0, tags.size); // 7 0 2
```

以下片段来自独立的 type-errors.ts：

```typescript
const scores = new Map<string, number>();
scores.set("课", "3"); // 值必须为 number。
const words = new Set<string>();
words.add(3); // 元素必须为 string。
async function wrongAsync(): number { return 3; } // async 的返回类型必须使用 Promise。
```

## 6 Iterable、Iterator 与 Generator

Iterable&lt;T&gt; 表达能通过 Symbol.iterator 获取迭代器的对象；Iterator&lt;T, TReturn, TNext&gt; 描述 next 等迭代方法。T 是每次产出的类型，TReturn 是完成时的返回类型，TNext 是恢复执行时送入的类型。只有 next 的对象不一定可用于 for...of。

Generator 同时可迭代并具有迭代器能力。下面 yield 产出 number，最终 return 返回 string，恢复时接收 number。第一次 next 启动生成器，传入的值不会进入尚未执行的 yield；第二次 next(3) 才为暂停的表达式提供值。检查 done 后才能把迭代结果的 value 按产出或返回分支使用。

```typescript
function sumIterable(values: Iterable<number>): number {
  let sum = 0;
  for (const value of values) sum += value;
  return sum;
}
function* exchange(): Generator<number, string, number> {
  const added: number = yield 2;
  return "sum:" + (2 + added);
}
const iterator: Iterator<number, string, number> = exchange();
const start = iterator.next();
const end = iterator.next(3);
const endText = end.done ? end.value.toUpperCase() : String(end.value);
console.log(sumIterable(new Set([2, 3])), JSON.stringify(start), endText); // 5 {"value":2,"done":false} SUM:5
```

以下片段来自独立的 type-errors.ts：

```typescript
function* numericInput(): Generator<number, void, number> { const added: number = yield 1; }
const iterator = numericInput();
iterator.next("3"); // 恢复输入 TNext 为 number。
const nextOnly: Iterator<number, void, unknown> = { next: () => ({ done: true, value: undefined }) };
const iterable: Iterable<number> = nextOnly; // 缺少 Symbol.iterator。
```

## 7 异步迭代类型与结束

AsyncIterator 的 next 返回迭代结果的 Promise；AsyncIterable 通过 Symbol.asyncIterator 提供异步迭代器。AsyncGenerator 则组合两种能力，也分别描述产出、最终返回和恢复输入类型。

for await...of 消费逐次产出的值，不把生成器最终 return 当作一个元素。所有 await 都需要实际等待完成；本例产出两个字符串并正常结束，没有计时器、服务或外部资源。Promise 类型和迭代协议的作用不同，不应把 Promise&lt;string[]&gt; 与 AsyncIterable&lt;string&gt; 当作同一种接口。

```typescript
async function* rows(): AsyncGenerator<string, void, unknown> {
  yield "A";
  yield "B";
}
async function collect<T>(source: AsyncIterable<T>): Promise<T[]> {
  const values: T[] = [];
  for await (const value of source) values.push(value);
  return values;
}
const collected = await collect(rows());
const asyncIterator: AsyncIterator<string, void, unknown> = rows();
await asyncIterator.next();
await asyncIterator.next();
const finished = await asyncIterator.next();
console.log(collected.join(","), finished.done); // A,B true
```

## 8 型变与 in、out 标注

型变（variance）描述类型实参之间的关系如何影响同一泛型结构的兼容性。生产 T 的结构通常协变：生产具体类型也能作为生产更宽类型；消费 T 的函数结构通常逆变：能消费更宽输入就能消费更窄输入。既消费又生产时可能需要不变关系。

in 标记逆变，out 标记协变，in out 表达不变。它们仅适用于受支持的泛型类型声明，并在同一泛型的实例化比较中起作用，不会强迫任意结构比较改变规则。应与实际结构方向相符；日常代码通常让编译器推断即可，不靠标注修补不安全接口。

```typescript
interface Source<out T> { read: () => T; }
interface Sink<in T> { write: (value: T) => void; }
interface Cell<in out T> { read: () => T; write: (value: T) => void; }
const source: Source<string> = { read: () => "TS" };
const widerSource: Source<string | number> = source;
const consumed: (string | number)[] = [];
const widerSink: Sink<string | number> = { write: (value) => { consumed.push(value); } };
const textSink: Sink<string> = widerSink;
textSink.write("类型");
const cell: Cell<number> = { read: () => 1, write: (value) => { consumed.push(value); } };
console.log(widerSource.read(), consumed.join(","), cell.read()); // TS 类型 1
```

以下片段来自独立的 type-errors.ts：

```typescript
interface Source<out T> { read: () => T; }
const broadSource: Source<string | number> = { read: () => 3 };
const tooNarrow: Source<string> = broadSource; // 可能产出 number，不满足只产出 string 的契约。
interface WrongVariance<out T> { write: (value: T) => void; } // 实际消费 T，不能声明为协变。
```

## 本章小结

泛型应表达已有的类型关系，约束只提供最小能力。const 推断和元组展开保留更具体结构；集合、Promise 与迭代器的类型参数各有职责。型变标注不改变运行时或任意结构比较规则。

## 练习

1. 写一个泛型映射函数，把 readonly T[] 与 (value: T) =&gt; U 转成 U[]；T 表示输入元素、U 表示输出元素。数值转字符串后结果应可赋给 string[]，空输入应返回空数组。
2. 写生成器，产出两次数值、最终返回字符串；通过 done 分支使用结果，不能把最终字符串计入 for...of 的数值和。
3. 把异步 collect 应用于空源与两项源，分别得到空数组和原顺序数组；等所有 Promise 完成后再退出。
4. 为生产者和消费者各给一对安全、危险的赋值，核对参数方向，解释为何不需要为了常规使用手写型变标注。

### 提示

1. 泛型签名连接数组元素 T、回调输入 T 与回调输出 U，直接使用 map。
2. 分别创建两个生成器实例用于 next 观察与 for...of；不要重复使用已消费完的实例。
3. 每次 collect 接收新建的异步源，调用时 await。
4. 分别从 read 的结果与 write 的输入判断替换方向。


### 参考解析

1. 函数体可以直接返回 values.map(convert)；例如 [1,2] 经 String 得 ["1","2"]，空数组得 []，无需额外空数组分支。
2. 如依次 yield 2、yield 3、return "done"，前三次 next 的 done 为 false、false、true；独立 for...of 实例只累加 2 与 3，结果为 5。
3. 空异步源得到 []，按顺序产出 A、B 的源得到 ["A","B"]；外层 Promise 完成才表示收集结束。
4. Source&lt;string&gt; 可赋给 Source&lt;string | number&gt;，反向不成立；Sink 的安全方向相反。结构已经表达读写关系时，通常无需显式 in/out。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [Generics：函数、接口、约束、默认与 Variance Annotations](https://www.typescriptlang.org/docs/handbook/2/generics.html)；[Object Types：标准库泛型容器](https://www.typescriptlang.org/docs/handbook/2/objects.html#the-array-type)；[Everyday Types：返回 Promise 的函数](https://www.typescriptlang.org/docs/handbook/2/everyday-types.html#functions-which-return-promises)；[4.0：Variadic Tuple Types](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-4-0.html#variadic-tuple-types)；[5.0：const 类型参数](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-5-0.html#const-type-parameters)；[3.6：Stricter Generators](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-3-6.html#stricter-generators)；[2.3：Async Iteration（类型引入历史；本章使用当前 ES2025 声明）](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-2-3.html#async-iteration)；[迭代器与生成器](https://www.typescriptlang.org/docs/handbook/iterators-and-generators.html)；[noUncheckedIndexedAccess](https://www.typescriptlang.org/tsconfig/noUncheckedIndexedAccess.html)；[4.7：可选型变标注](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-4-7.html#optional-variance-annotations-for-type-parameters)。 |
| ECMA-262 第 16 版（ECMAScript 2025） | [24.1.3.6：Map.get 的缺失结果](https://262.ecma-international.org/16.0/#sec-map.prototype.get)；[24.2.4.1：Set.add 的去重](https://262.ecma-international.org/16.0/#sec-set.prototype.add)；[27.5.1.2：生成器 next](https://262.ecma-international.org/16.0/#sec-generator.prototype.next)。 |
| npm 官方文档 | [npm run（v11）](https://docs.npmjs.com/cli/v11/commands/npm-run/)：从本技术目录运行已配置脚本，并解析本地工具。 |
